# Audio Synchronization
This notebook explore several Python libraries for audio synchronization of 2 audio tracks.

New dependencies: numpy, soundfile, scipy, librosa

### Requirements

File must be an audio (e.g., mp3, wav), video files must be converted.

In [ ]:
import numpy as np
import soundfile as sf
from scipy.signal import correlate, correlation_lags
from IPython.display import display, Audio # not needed for production
from pathlib import Path

def load_mono(path):
    audio, sr = sf.read(path)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    audio = audio.astype(np.float32)
    audio -= np.mean(audio)
    audio /= np.max(np.abs(audio)) + 1e-9
    return audio, sr

import subprocess
def convert_sample_rate(input_path, output_path, target_sr):
    subprocess.run(
        ["ffmpeg", "-y", "-i", input_path, "-ar", str(target_sr), output_path]
    )
    return output_path


def estimate_offset_seconds(reference_path, target_path):
    """Audio by be the same sample rate. MP3 input seems to be fine, but sample rate must match."""
    ref, sr1 = load_mono(reference_path)
    target, sr2 = load_mono(target_path)

    if sr1 != sr2:
        print(f"Sample rates differ: {sr1} vs {sr2}. Converting target to {sr1}.")
        converted_path = convert_sample_rate(target_path, Path(target_path).with_suffix(".converted.wav"), sr1)
        target, sr2 = load_mono(converted_path)

    corr = correlate(target, ref, mode="full", method="fft")
    lags = correlation_lags(len(target), len(ref), mode="full")

    best_lag = lags[np.argmax(corr)]
    return best_lag / sr1

In [53]:
clean_bg_path = "synchronization/cool_bg.mp3"
karaoke_path =  "synchronization/cool_karaoke.mp3"

In [54]:
estimated_offset = estimate_offset_seconds(clean_bg_path, karaoke_path)
print(f"Estimated offset: {estimated_offset:.2f} seconds")

Sample rates differ: 44100 vs 48000. Converting target to 44100.
Estimated offset: 6.92 seconds


Positive value: target is ahead of reference

Negative value: target is behind of reference

Remake the vocals with the synchronized timing.

In [ ]:
"""
This is intended to be run on the backend (not demucs service), ideally, the demucs service should only return a timestamp offset and a vocals file. Since in the frontend, we can still allow the user to customize the offset before committing the cut.
"""

import json
import shutil
import subprocess
import tempfile
from pathlib import Path


ffmpeg_path = r"C:\Partitions\G\youtube_dlp\ffmpeg.exe"
ffprobe_path = r"C:\Partitions\G\youtube_dlp\ffprobe.exe"


def _run(cmd):
    subprocess.run(cmd, check=True)


def _probe_duration(path: str) -> float:
    out = subprocess.check_output([
        ffprobe_path,
        "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        path,
    ])
    return float(out.decode().strip())


def _probe_audio_stream(path: str) -> dict:
    out = subprocess.check_output([
        ffprobe_path,
        "-v", "error",
        "-select_streams", "a:0",
        "-show_entries", "stream=sample_rate,channels,bit_rate",
        "-of", "json",
        path,
    ])
    data = json.loads(out.decode())
    if not data.get("streams"):
        raise ValueError(f"No audio stream found: {path}")
    return data["streams"][0]


def _concat_file_line(path: str) -> str:
    # ffmpeg concat demuxer likes forward slashes on Windows
    p = str(Path(path).resolve()).replace("\\", "/")
    p = p.replace("'", "'\\''")
    return f"file '{p}'\n"


def _write_concat_list(paths, list_path: str):
    with open(list_path, "w", encoding="utf-8") as f:
        for p in paths:
            f.write(_concat_file_line(p))


def _generate_silence_mp3(
    output_path: str,
    duration: float,
    sample_rate: int,
    channels: int,
    bitrate: str | None = None,
):
    if duration <= 0:
        raise ValueError("Silence duration must be positive")

    channel_layout = "mono" if channels == 1 else "stereo"

    cmd = [
        ffmpeg_path,
        "-y",
        "-f", "lavfi",
        "-i", f"anullsrc=r={sample_rate}:cl={channel_layout}",
        "-t", str(duration),
        "-c:a", "libmp3lame",
    ]

    if bitrate:
        cmd += ["-b:a", bitrate]

    cmd += [output_path]
    _run(cmd)


def _copy_mp3_segment(
    input_path: str,
    output_path: str,
    start: float = 0.0,
    duration: float | None = None,
):
    cmd = [ffmpeg_path, "-y"]

    if start > 0:
        cmd += ["-ss", str(start)]

    cmd += ["-i", input_path]

    if duration is not None:
        cmd += ["-t", str(duration)]

    cmd += ["-c", "copy", output_path]
    _run(cmd)


def _concat_mp3_copy(paths, output_path: str, duration: float | None = None):
    with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False, encoding="utf-8") as f:
        list_path = f.name

    try:
        _write_concat_list(paths, list_path)

        cmd = [
            ffmpeg_path,
            "-y",
            "-f", "concat",
            "-safe", "0",
            "-i", list_path,
        ]

        if duration is not None:
            cmd += ["-t", str(duration)]

        cmd += ["-c", "copy", output_path]
        _run(cmd)

    finally:
        Path(list_path).unlink(missing_ok=True)


def remake_vocals_concat(
    vocals_path: str,
    karaoke_path: str,
    output_path: str,
    offset_seconds: float,
) -> str:
    """
    Creates an aligned standalone MP3 vocal file.

    offset_seconds > 0:
        vocals are ahead of karaoke, so prepend silence.

    offset_seconds < 0:
        vocals are behind karaoke, so trim from the beginning.

    The original vocal MP3 frames are copied where possible.
    Silence segments are newly encoded.
    """

    karaoke_duration = _probe_duration(karaoke_path)
    vocals_duration = _probe_duration(vocals_path)

    stream = _probe_audio_stream(vocals_path)
    sample_rate = int(stream["sample_rate"])
    channels = int(stream["channels"])

    bitrate = stream.get("bit_rate")
    bitrate_arg = f"{round(int(bitrate) / 1000)}k" if bitrate else None

    offset_seconds = float(offset_seconds)

    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)

        # Case 1: no offset, just copy/trim to karaoke duration
        if abs(offset_seconds) < 0.001:
            _copy_mp3_segment(
                vocals_path,
                output_path,
                start=0,
                duration=karaoke_duration,
            )
            return output_path

        # Case 2: vocals are ahead, prepend silence
        if offset_seconds > 0:
            silence_path = str(tmp / "silence.mp3")

            _generate_silence_mp3(
                output_path=silence_path,
                duration=offset_seconds,
                sample_rate=sample_rate,
                channels=channels,
                bitrate=bitrate_arg,
            )

            _concat_mp3_copy(
                [silence_path, vocals_path],
                output_path,
                duration=karaoke_duration,
            )

            return output_path

        # Case 3: vocals are behind, trim the beginning
        shift = abs(offset_seconds)

        if shift >= vocals_duration:
            _generate_silence_mp3(
                output_path=output_path,
                duration=karaoke_duration,
                sample_rate=sample_rate,
                channels=channels,
                bitrate=bitrate_arg,
            )
            return output_path

        trimmed_path = str(tmp / "trimmed.mp3")

        _copy_mp3_segment(
            vocals_path,
            trimmed_path,
            start=shift,
            duration=karaoke_duration,
        )

        trimmed_duration = _probe_duration(trimmed_path)

        # If trimmed vocal is shorter than karaoke, append silence
        if trimmed_duration < karaoke_duration - 0.05:
            tail_silence = str(tmp / "tail_silence.mp3")
            remaining = karaoke_duration - trimmed_duration

            _generate_silence_mp3(
                output_path=tail_silence,
                duration=remaining,
                sample_rate=sample_rate,
                channels=channels,
                bitrate=bitrate_arg,
            )

            _concat_mp3_copy(
                [trimmed_path, tail_silence],
                output_path,
                duration=karaoke_duration,
            )
        else:
            shutil.copyfile(trimmed_path, output_path)

        return output_path

In [58]:
remaked_vocals_path = remake_vocals_concat(
    vocals_path="synchronization/cool_vocals.mp3",
    karaoke_path=karaoke_path,
    output_path="synchronization/remaked_vocals.mp3",
    offset_seconds=estimated_offset
)

Using Librosa

We can optionally use librosa in case the previous method does not work well. This is probably the retry flow. It's the same as the previous method, we just replace the cross-correlation with DTW.

In [44]:
import librosa
import numpy as np

def estimate_offset_with_librosa(
    karaoke_bg_path: str,
    extracted_bg_path: str,
    sr: int = 22050,
) -> float:
    # Load both audio files as mono
    karaoke, _ = librosa.load(karaoke_bg_path, sr=sr, mono=True)
    extracted, _ = librosa.load(extracted_bg_path, sr=sr, mono=True)

    # Optional: trim silence
    karaoke, _ = librosa.effects.trim(karaoke, top_db=30)
    extracted, _ = librosa.effects.trim(extracted, top_db=30)

    # Convert to chroma features
    karaoke_chroma = librosa.feature.chroma_cqt(y=karaoke, sr=sr)
    extracted_chroma = librosa.feature.chroma_cqt(y=extracted, sr=sr)

    # DTW alignment
    D, wp = librosa.sequence.dtw(
        X=karaoke_chroma,
        Y=extracted_chroma,
        metric="cosine"
    )

    # wp is the warping path: pairs of frame indexes
    # Reverse it so it goes forward in time
    wp = np.array(wp[::-1])

    karaoke_frames = wp[:, 0]
    extracted_frames = wp[:, 1]

    # Convert frames to time
    karaoke_times = librosa.frames_to_time(karaoke_frames, sr=sr)
    extracted_times = librosa.frames_to_time(extracted_frames, sr=sr)

    # Estimate offset between the two timelines
    offsets = extracted_times - karaoke_times

    # Use median to avoid outliers
    offset_seconds = float(np.median(offsets))

    return offset_seconds

In [51]:
clean_bg_path = "synchronization/opalite_bg.mp3"
karaoke_path =  "synchronization/opalite_karaoke.mp3"
estimated_offset = estimate_offset_with_librosa(clean_bg_path, karaoke_path)
estimated_offset

0.0